In [ ]:
import pandas as pd
import os

from scoring import split_exam, seperate_scoring, kice_metric

In [ ]:
result = pd.read_csv("/Users/jinminseong/Desktop/KICE_slayer_AI_Korean/scored_result_gpt-5.1-2025-11-13.csv")

In [ ]:
import re
# 폴더 내의 모든 파일 목록 가져오기
exp_root = "autorag_project_dir/27/prompt_node_line/generator"
all_files = os.listdir(exp_root)

# 숫자만 있는 파일명과 .parquet 확장자를 가진 파일 필터링
parquet_files = [f for f in all_files if re.match(r'^\d+\.parquet$', f)]

In [ ]:
all_files

In [ ]:
pd.read_csv(exp_root + '/summary.csv')['module_params'][0]

In [ ]:
generated_answer_lst = []

project_file_dir = f"autorag_project_dir/{27}/prompt_node_line/generator"
all_files = os.listdir(project_file_dir)


for idx, f in enumerate(all_files):
    if re.match(r'^\d+\.parquet$', f):
        model_name = experience_summary.loc[experience_summary['filename'] == f, 'module_params'].values
        df_result = pd.read_parquet(os.path.join(project_file_dir, f)).drop(columns=['kice_metric'])
        print(df_result)

In [ ]:
test_0 = pd.read_parquet(os.path.join(project_file_dir, '0.parquet'))
test_1 = pd.read_parquet(os.path.join(project_file_dir, '1.parquet'))
best_1 = pd.read_parquet(os.path.join(project_file_dir, 'best_1.parquet'))

In [ ]:
best_1

In [ ]:
assert len(test_0) == len(test_1) == len(best_1)

In [ ]:
def main(exp_num: int):
    results_lst = []
    project_file_dir = f"autorag_project_dir/{exp_num}/prompt_node_line/generator"

    result = pd.read_parquet(f"autorag_project_dir/{exp_num}/prompt_node_line/generator/best_0.parquet").drop(
        columns=['kice_metric'])
    generated_answer = kice_metric(result)

    result = pd.concat([result, pd.DataFrame(generated_answer, columns=['kice_metric'])], axis=1)
    result.to_csv(f'scored_result_{exp_num}.csv', index=False)

    result = split_exam(result)
    overall = seperate_scoring(result)
    overall.to_csv("scoring_result/Qwen2-72B-Instruct.csv")
    logging.info(sum(seperate_scoring(overall)['overall_sum']) / 10)

In [ ]:
def split_exam(df):
    df['qid_for_merge'] = df['qid'].apply(lambda x: x[:4])
    merged_df = df.groupby('qid_for_merge').agg({
        'kice_metric': list
    }).reset_index()
    return merged_df

In [ ]:
test = split_exam(result)

In [ ]:
def seperate_scoring(df):
    def seperate(anual_row_lst):
        common_problem_score = 0
        choice_problem_score = 0
        test_cnt = 0

        for idx, score in enumerate(anual_row_lst):
            if idx < 34:
                common_problem_score += score
                test_cnt += 1
            else:
                choice_problem_score += score
        assert test_cnt == 34

        return {'common_problem_score': common_problem_score, 'choice_problem_score': choice_problem_score, 'overall_sum': common_problem_score+choice_problem_score}

    df['score_result'] = df.apply(
        lambda row: {'common_problem_score': sum(row['kice_metric']), 'choice_problem_score': None, 'overall_sum': sum(row['kice_metric'])} if int(
            row['qid_for_merge']) < 2022 else seperate(row['kice_metric']), axis=1)
    result = pd.concat([df, df['score_result'].apply(pd.Series)], axis=1)
    result = result.rename(columns={0: 'common_problem_score', 1: 'choice_problem_score'})
    return result

In [ ]:
overall = seperate_scoring(test)

In [ ]:
overall

In [ ]:
overall.to_csv("scoring_result/57_gpt-4o.csv")

In [ ]:
sum(seperate_scoring(test)['overall_sum'])/10

In [ ]:
pd.read_parquet(f"autorag_project_dir/27/prompt_node_line/generator/0.parquet")

In [ ]:
pd.read_csv("autorag_project_dir/27/prompt_node_line/generator/summary.csv")['module_params'][1]

# 등급매기는건 자동화 필요

# 평균과 점수 계산후 등급을 바로 table로 만들어주는 GPT 개발

# 규식이형으로 등급과 모델이름 그리고 각 년도별 수능 자동 입력기

# markdown table에서 parsing후 assert 진행

# Crux Table 개발하기

### 필요한것
1. 백분위
2. 표준편차
3. 등급
4. 비율
5. 성적통지표 개발
6. 10개년 벤치마킹용 데이터셋 구축
[출처: 오르비 crux table, 종로학원, 대교협]
-> 각 입시학원마다 원점수 등급컷이 다른 이유는 평가원은 표준점수만을 토대로 입시학원 본인들만의 수치를 통해서 역산한것이므로 학원마다 가지고 있는 수치가 달라서 불가능

In [ ]:
project_file_dir = 'scoring_result/'
all_files = os.listdir(project_file_dir)
for file in all_files:
    drop_unnamed_col = pd.read_csv(os.path.join(project_file_dir, file)).drop(columns=['Unnamed: 0'])
    drop_unnamed_col.to_csv(os.path.join(project_file_dir, file), index=False)

In [ ]:
all_files

In [ ]:
import pandas as pd
tmp = pd.read_csv('legacy_result/scoring_result/claude-3-5-sonnet-20241022.csv')
tmp

In [ ]:
pd.read_csv(f"completion_table_ver2.csv")

In [ ]:
import pandas as pd
pd.read_csv('/Users/jinminseong/Desktop/KICE_slayer_AI_Korean/scored_result_gpt-4.1-2025-04-14.csv')['kice_metric'].sum()

In [ ]:
df = pd.read_csv('/Users/jinminseong/Desktop/KICE_slayer_AI_Korean/scored_result_Kimi-K2-Thinking.csv')